In [0]:
%pip install catboost openpyxl

In [0]:
from pathlib import Path
import pandas as pd

# Datenordner aus deinem aktuellen NMF-Notebook
basis = Path("/Workspace/Users/khalil-said.albert@de.abb.com/Daten")

# Passende Excel-Dateien suchen
bom_dateien = sorted(basis.glob("*_korrigiert.xlsx"))

# Benötigte Dateien prüfen
datei_check = pd.DataFrame([
    {"Prüfung": "bom_core.py vorhanden",
     "OK": (basis / "bom_core.py").is_file()},
    {"Prüfung": "layer2_profile.py vorhanden",
     "OK": (basis / "layer2_profile.py").is_file()},
    {"Prüfung": "Genau eine korrigierte Stückliste vorhanden",
     "OK": len(bom_dateien) == 1},
])

display(datei_check)

In [0]:
import sys
import time

assert datei_check["OK"].all(), "Bitte zuerst die fehlenden Dateien prüfen."

# Python soll eure Module im Datenordner finden
if str(basis) not in sys.path:
    sys.path.insert(0, str(basis))

import bom_core as bc
import layer2_profile as L2

# Die zuvor gefundene Stückliste mit dem bestehenden Cleaner laden
arbeitsdatei = str(bom_dateien[0])
start = time.perf_counter()

sauber, protokoll = bc.clean_bom(arbeitsdatei)

display(pd.DataFrame([{
    "Positionszeilen": len(sauber),
    "Erwartete Positionszeilen": 47390,
    "Stand stimmt überein": len(sauber) == 47390,
    "Laufzeit in Sekunden": round(time.perf_counter() - start, 1)
}]))

In [0]:
# Aus der Liste eine Tabelle mit den drei benötigten Spalten bilden
positionen = pd.DataFrame(sauber)[
    ["erzeugnis", "block", "komponente"]
].copy()

assert positionen.notna().all().all(), "Produkt, Block oder Komponente fehlt."

# Pro Produktblock die verschiedenen Komponenten zusammenstellen
block_sets = (
    positionen.groupby(["erzeugnis", "block"])["komponente"]
    .agg(lambda komponenten: tuple(sorted(set(komponenten))))
    .reset_index(name="komponentenset")
)

# Pro Produkt die Blöcke und unterschiedlichen Komponentensets zählen
block_check = (
    block_sets.groupby("erzeugnis")
    .agg(
        bloecke=("block", "size"),
        unterschiedliche_sets=("komponentenset", "nunique")
    )
)

display(pd.DataFrame([{
    "Produkte": len(block_check),
    "Produktblöcke": len(block_sets),
    "Produkte mit mehreren Blöcken": int((block_check["bloecke"] > 1).sum()),
    "Davon mit unterschiedlichen Komponentensets":
        int((block_check["unterschiedliche_sets"] > 1).sum())
}]))

In [0]:
# Die zwei Produkte mit unterschiedlichen Komponentensets auswählen
abweichende_produkte = block_check.index[
    block_check["unterschiedliche_sets"] > 1
]

unterschiede = []

for produkt in abweichende_produkte:
    fundstellen = block_sets[block_sets["erzeugnis"] == produkt]

    # Komponenten, die an jeder Fundstelle dieses Produkts vorkommen
    gemeinsam = set.intersection(
        *(set(teile) for teile in fundstellen["komponentenset"])
    )

    for _, zeile in fundstellen.iterrows():
        teile = set(zeile["komponentenset"])
        unterschiede.append({
            "Produkt": produkt,
            "Fundstelle": zeile["block"],
            "Anzahl Komponenten": len(teile),
            "Davon an allen Fundstellen vorhanden": len(gemeinsam),
            "Komponenten außerhalb des gemeinsamen Sets":
                ", ".join(map(str, sorted(teile - gemeinsam)))
        })


display(pd.DataFrame(unterschiede))

In [0]:
# Gleiche Komponentensets innerhalb desselben Produkts einmal zählen
fassungen = (
    block_sets[["erzeugnis", "komponentenset"]]
    .drop_duplicates(["erzeugnis", "komponentenset"])
    .reset_index(drop=True)
)

# Jede unterschiedliche Fassung bekommt eine technische Kennung
fassungen["fassung_id"] = fassungen.index

# Alle Fassungen eines Produkts erhalten zusammen das Gewicht 1
anzahl_fassungen = (
    fassungen.groupby("erzeugnis")["fassung_id"].transform("count")
)
fassungen["produktgewicht"] = 1.0 / anzahl_fassungen

display(pd.DataFrame([{
    "Produkte": fassungen["erzeugnis"].nunique(),
    "Unterschiedliche Fassungen": len(fassungen),
    "Produkte mit mehreren Fassungen":
        fassungen.loc[anzahl_fassungen > 1, "erzeugnis"].nunique(),
    "Summe der Produktgewichte": fassungen["produktgewicht"].sum()
}]))

In [0]:
# Hauptgruppe aus der Erzeugnisnummer ableiten (erste 6 Zeichen)
fassungen_mit_gruppen = fassungen.assign(
    hauptgruppe=fassungen["erzeugnis"].apply(L2.hauptgruppe_von)
)

# Jede Fassung muss einer Hauptgruppe zugeordnet sein
assert fassungen_mit_gruppen["hauptgruppe"].notna().all()

X_je_hauptgruppe = {}
info_je_hauptgruppe = {}

for hauptgruppe, gruppe in fassungen_mit_gruppen.groupby("hauptgruppe"):
    # Aus jedem Komponentenset eine Zeile je enthaltener Komponente machen
    vorhanden = (
        gruppe[["fassung_id", "komponentenset"]]
        .explode("komponentenset")
        .rename(columns={"komponentenset": "komponente"})
    )

    # Anwesenheit als 0/1-Tabelle darstellen
    X = pd.crosstab(vorhanden["fassung_id"], vorhanden["komponente"])
    X = X.reindex(index=gruppe["fassung_id"], fill_value=0)
    X = (X > 0).astype("int8")

    X_je_hauptgruppe[hauptgruppe] = X
    info_je_hauptgruppe[hauptgruppe] = (
        gruppe.set_index("fassung_id").loc[X.index].copy()
    )

display(pd.DataFrame([
    {
        "Hauptgruppe": hg,
        "Produkte": info_je_hauptgruppe[hg]["erzeugnis"].nunique(),
        "Fassungen": X.shape[0],
        "Verschiedene Komponenten": X.shape[1]
    }
    for hg, X in X_je_hauptgruppe.items()
]))

In [0]:
import numpy as np
import pandas as pd

HAUPTGRUPPE = "GJL121"
SEED = 42
N_FOLDS = 5

X_pilot = X_je_hauptgruppe[HAUPTGRUPPE].copy()
info_pilot = info_je_hauptgruppe[HAUPTGRUPPE].loc[X_pilot.index].copy()

# Produkte reproduzierbar mischen und auf fünf Prüfgruppen verteilen.
produkte = np.array(sorted(info_pilot["erzeugnis"].unique()))
np.random.default_rng(SEED).shuffle(produkte)

fold_je_produkt = {
    produkt: nummer % N_FOLDS
    for nummer, produkt in enumerate(produkte)
}
info_pilot["fold"] = info_pilot["erzeugnis"].map(fold_je_produkt)

# Alle Fassungen eines Produkts müssen im selben Fold liegen.
assert info_pilot.groupby("erzeugnis")["fold"].nunique().eq(1).all()
assert info_pilot.index.equals(X_pilot.index)

uebersicht = info_pilot.groupby("fold").agg(
    Pruefprodukte=("erzeugnis", "nunique"),
    Prueffassungen=("erzeugnis", "size"),
).reset_index()

uebersicht["Trainingsprodukte"] = len(produkte) - uebersicht["Pruefprodukte"]
display(uebersicht)

In [0]:
# Jede Fassung wird mit jeder Komponente kombiniert.
paare = (
    X_pilot.rename_axis(index="fassung_id", columns="komponente")
    .stack()
    .rename("vorhanden")
    .reset_index()
)

# Produkt, kleinere Gruppe, Gewicht und Fold zuordnen.
metadaten = (
    info_pilot[
        ["erzeugnis", "produktgewicht", "fold"]
    ]
    .rename_axis("fassung_id")
    .reset_index()
)

paare = paare.merge(
    metadaten, on="fassung_id", how="left", validate="many_to_one"
)

assert len(paare) == X_pilot.shape[0] * X_pilot.shape[1]
assert not paare.duplicated(["fassung_id", "komponente"]).any()
assert paare["fold"].notna().all()

display(pd.DataFrame([{
    "Prüfpaare": len(paare),
    "Davon vorhanden": int(paare["vorhanden"].sum()),
    "Davon nicht vorhanden": int(paare["vorhanden"].eq(0).sum()),
    "Verschiedene Komponenten": paare["komponente"].nunique(),
    "Fassungen": paare["fassung_id"].nunique()
}]))

In [0]:
# Für jedes Prüfpaar die Komponenten seiner Fassung übernehmen.
kontext = X_pilot.loc[paare["fassung_id"]].to_numpy(copy=True)
ziel_spalten = X_pilot.columns.get_indexer(paare["komponente"])
assert (ziel_spalten >= 0).all()

# Richtige Antworten separat sichern und mit der Quelle vergleichen.
y_paare = paare["vorhanden"].to_numpy(copy=True)
zeilen = np.arange(len(paare))
assert np.array_equal(kontext[zeilen, ziel_spalten], y_paare)

# Die eigene Anwesenheit darf keine Modelleingabe sein.
kontext[zeilen, ziel_spalten] = 0

features_paare = pd.DataFrame(
    kontext,
    columns=[f"kontext_{i}" for i in range(X_pilot.shape[1])],
    index=paare.index
)

features_paare["pruefkomponente"] = paare["komponente"].astype(str)
kategoriale_spalten = ["pruefkomponente"]

display(pd.DataFrame([{
    "Zeilen": len(features_paare),
    "Eingabespalten": features_paare.shape[1],
    "Prüfkomponenten vollständig ausgeblendet":
        bool((kontext[zeilen, ziel_spalten] == 0).all()),
    "Lernantwort unverändert":
        bool(np.array_equal(y_paare, paare["vorhanden"].to_numpy()))
}]))

In [0]:
from catboost import CatBoostClassifier

MIN_PRODUKTE = 5
modelle = {}
modell_spalten = {}
laufbericht = []

assert features_paare.index.equals(paare.index)
assert np.array_equal(y_paare, paare["vorhanden"].to_numpy())
assert paare.groupby("erzeugnis")["fold"].nunique().eq(1).all()

ergebnisse = paare.copy()
ergebnisse["p_anwesend"] = np.nan
ergebnisse["pruefstatus"] = "noch_nicht_geprueft"

for fold in sorted(paare["fold"].unique()):
    train = paare["fold"].ne(fold)
    pruefen = paare["fold"].eq(fold)
    trainingspaare = paare.loc[train]

    # Nur Trainingsprodukte zählen, jede Produktnummer höchstens einmal je Antwort.
    anzahl = (
        trainingspaare.groupby(["komponente", "vorhanden"])["erzeugnis"]
        .nunique().unstack(fill_value=0)
        .reindex(index=X_pilot.columns, columns=[0, 1], fill_value=0)
    )

    status = pd.Series("modell_bewertbar", index=anzahl.index)
    status.loc[(anzahl[0] < MIN_PRODUKTE) |
               (anzahl[1] < MIN_PRODUKTE)] = "wenig_trainingsbelege"
    status.loc[anzahl[0].eq(0)] = "im_training_immer_anwesend"
    status.loc[anzahl[1].eq(0)] = "im_training_nie_anwesend"

    ergebnisse.loc[pruefen, "pruefstatus"] = (
        paare.loc[pruefen, "komponente"].map(status)
    )

    geeignete = status.index[status.eq("modell_bewertbar")]
    fit_maske = train & paare["komponente"].isin(geeignete)
    score_maske = pruefen & paare["komponente"].isin(geeignete)

    # Kontextspalten nur für im Trainingsbestand vorhandene Komponenten.
    spalten = [
        f"kontext_{i}"
        for i, komponente in enumerate(X_pilot.columns)
        if anzahl.loc[komponente, 1] > 0
    ] + ["pruefkomponente"]

    if fit_maske.any():
        modell = CatBoostClassifier(
            iterations=300,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            random_seed=42 + int(fold),
            thread_count=4,
            verbose=False,
            allow_writing_files=False
        )

        modell.fit(
            features_paare.loc[fit_maske, spalten],
            y_paare[fit_maske.to_numpy()],
            cat_features=["pruefkomponente"],
            sample_weight=paare.loc[fit_maske, "produktgewicht"].to_numpy()
        )

        if score_maske.any():
            ergebnisse.loc[score_maske, "p_anwesend"] = (
                modell.predict_proba(
                    features_paare.loc[score_maske, spalten]
                )[:, 1]
            )

        modelle[fold] = modell
        modell_spalten[fold] = spalten

    laufbericht.append({
        "Fold": fold,
        "Trainingspaare": int(fit_maske.sum()),
        "Bewertbare Komponenten": len(geeignete),
        "Bewertete Prüfpaare":
            int(ergebnisse.loc[pruefen, "p_anwesend"].notna().sum()),
        "Separat zu prüfende Paare": int((pruefen & ~score_maske).sum())
    })

assert len(ergebnisse) == len(paare)
assert ergebnisse["pruefstatus"].ne("noch_nicht_geprueft").all()
assert ergebnisse.loc[
    ergebnisse["pruefstatus"].eq("modell_bewertbar"), "p_anwesend"
].notna().all()

display(pd.DataFrame(laufbericht))

In [0]:
ergebnisse["p_haeufigkeit"] = np.nan
vergleich = []

for fold in sorted(paare["fold"].unique()):
    # Vergleichswert ausschließlich aus den anderen vier Folds.
    training = paare.loc[paare["fold"].ne(fold)].copy()
    training["gewichtete_anwesenheit"] = (
        training["vorhanden"] * training["produktgewicht"]
    )

    statistik = training.groupby("komponente").agg(
        anwesend=("gewichtete_anwesenheit", "sum"),
        gesamt=("produktgewicht", "sum")
    )

    # Je ein gedachter positiver und negativer Beleg vermeidet 0 und 1.
    haeufigkeit = (
        (statistik["anwesend"] + 1) / (statistik["gesamt"] + 2)
    )

    pruefen = ergebnisse["fold"].eq(fold)
    ergebnisse.loc[pruefen, "p_haeufigkeit"] = (
        ergebnisse.loc[pruefen, "komponente"].map(haeufigkeit)
    )

    # Fairer Vergleich: nur Paare mit einer CatBoost-Vorhersage.
    bewertet = ergebnisse.loc[
        pruefen & ergebnisse["p_anwesend"].notna()
    ]
    assert not bewertet.empty
    assert bewertet["p_haeufigkeit"].notna().all()

    fehler = {}
    for name, spalte in [
        ("CatBoost", "p_anwesend"),
        ("Häufigkeit", "p_haeufigkeit")
    ]:
        fehler[name] = np.average(
            (bewertet[spalte] - bewertet["vorhanden"]) ** 2,
            weights=bewertet["produktgewicht"]
        )

    vergleich.append({
        "Fold": fold,
        "Verglichene Paare": len(bewertet),
        "Brier CatBoost": fehler["CatBoost"],
        "Brier Häufigkeit": fehler["Häufigkeit"],
        "Vorteil CatBoost": fehler["Häufigkeit"] - fehler["CatBoost"]
    })

display(pd.DataFrame(vergleich).round(5))

In [0]:
# Hoher Wert bedeutet: Modell-Erwartung und Stückliste widersprechen sich.
ergebnisse["auffaelligkeit"] = np.where(
    ergebnisse["vorhanden"].eq(0),
    ergebnisse["p_anwesend"],
    1 - ergebnisse["p_anwesend"]
)

ergebnisse["pruefrichtung"] = np.where(
    ergebnisse["vorhanden"].eq(0),
    "möglicherweise fehlend",
    "möglicherweise zusätzlich"
)

# Nur die Anzeige begrenzen; der vollständige Bestand bleibt erhalten.
vorschau = (
    ergebnisse.loc[ergebnisse["p_anwesend"].notna()]
    .sort_values("auffaelligkeit", ascending=False)
    .groupby("pruefrichtung", sort=False)
    .head(5)
)

display(vorschau[[
    "erzeugnis", "fassung_id", "komponente",
    "vorhanden", "p_anwesend", "auffaelligkeit", "pruefrichtung"
]].round(4))

In [0]:
import re

FEHLT_V2 = "OHNE_ANGABE"

def spannung_v2(text):
    t = str(text).replace("*", " ").replace("+", " ").replace("_", " ")
    treffer = re.findall(r"(\d+(?:[-/]\d+)?)\s*V\s*(DC|AC)?", t, re.I)
    werte = set()
    for wert, art in treffer:
        art = art.upper() or (
            "AC" if re.search(r"\d+(?:-\d+)?\s*HZ", t, re.I)
            else "UNBEKANNT"
        )
        werte.add(f"{wert}_{art}")
    return (
        next(iter(werte)) if len(werte) == 1
        else "MEHRDEUTIG" if werte else FEHLT_V2
    )

def eindeutig_v2(spalte):
    werte = set(spalte.fillna(FEHLT_V2))
    return next(iter(werte)) if len(werte) == 1 else "MEHRDEUTIG"

quelle_v2 = pd.DataFrame(sauber)[[
    "erzeugnis", "erzeugnis_txt", "komponente", "komponente_txt"
]].copy()

quelle_v2["erzeugnis"] = quelle_v2["erzeugnis"].astype(str)
quelle_v2["komponente"] = quelle_v2["komponente"].astype(str)
quelle_v2 = quelle_v2[
    quelle_v2["erzeugnis"].isin(paare["erzeugnis"].astype(str))
].copy()

quelle_v2["kleinere_gruppe"] = quelle_v2["erzeugnis_txt"].fillna("").map(
    lambda t: L2.typ_von(str(t)) or FEHLT_V2
)
quelle_v2["produkt_spannung"] = quelle_v2["erzeugnis_txt"].map(spannung_v2)
quelle_v2["komponente_spannung"] = quelle_v2["komponente_txt"].map(spannung_v2)

produkt_merkmale_v2 = quelle_v2.groupby("erzeugnis")[[
    "kleinere_gruppe", "produkt_spannung"
]].agg(eindeutig_v2)

komponenten_merkmale_v2 = quelle_v2.groupby("komponente")[[
    "komponente_spannung"
]].agg(eindeutig_v2)

komponenten_merkmale_v2["komponente_basis"] = (
    komponenten_merkmale_v2.index.to_series()
    .str.extract(r"^(.+?)[A-Za-z]\d{4}$", expand=False)
    .fillna(FEHLT_V2)
)

assert features_paare.index.equals(paare.index)
features_v2 = features_paare.copy()

for merkmal in produkt_merkmale_v2.columns:
    features_v2[merkmal] = paare["erzeugnis"].astype(str).map(
        produkt_merkmale_v2[merkmal]
    )

for merkmal in komponenten_merkmale_v2.columns:
    features_v2[merkmal] = paare["komponente"].astype(str).map(
        komponenten_merkmale_v2[merkmal]
    )

assert features_v2.notna().all().all(), "Eine Merkmalszuordnung fehlt."
assert features_v2.shape == (len(paare), features_paare.shape[1] + 4)

kategoriale_spalten_v2 = [
    "pruefkomponente", "kleinere_gruppe", "produkt_spannung",
    "komponente_spannung", "komponente_basis"
]

display(pd.DataFrame([
    {
        "Merkmal": merkmal,
        "Bezugsgröße": bezug,
        "Anzahl": len(tabelle),
        "Ohne Angabe": int(tabelle[merkmal].eq(FEHLT_V2).sum()),
        "Mehrdeutig": int(tabelle[merkmal].eq("MEHRDEUTIG").sum())
    }
    for tabelle, bezug in [
        (produkt_merkmale_v2, "Produkte"),
        (komponenten_merkmale_v2, "Komponenten")
    ]
    for merkmal in tabelle.columns
]))

In [0]:
from catboost import CatBoostClassifier

assert features_v2.index.equals(paare.index)
assert ergebnisse.index.equals(paare.index)
assert np.array_equal(y_paare, paare["vorhanden"].to_numpy())

neue_merkmale = [
    "kleinere_gruppe", "produkt_spannung",
    "komponente_spannung", "komponente_basis"
]

ergebnisse_v2 = ergebnisse.copy()
ergebnisse_v2["p_anwesend_v1"] = ergebnisse["p_anwesend"]
ergebnisse_v2["p_anwesend"] = np.nan

modelle_v2 = {}
modell_spalten_v2 = {}
vergleich_v2 = []

for fold in sorted(modelle):
    pruefen = paare["fold"].eq(fold) & ergebnisse["p_anwesend"].notna()

    # Dieselben Komponenten wie bei V1, dort anhand des Trainings ausgewählt.
    komponenten = paare.loc[pruefen, "komponente"].unique()
    trainieren = paare["fold"].ne(fold) & paare["komponente"].isin(komponenten)

    spalten = list(modell_spalten[fold]) + neue_merkmale
    parameter = modelle[fold].get_params().copy()
    parameter.pop("cat_features", None)

    modell = CatBoostClassifier(**parameter)
    modell.fit(
        features_v2.loc[trainieren, spalten],
        y_paare[trainieren.to_numpy()],
        cat_features=kategoriale_spalten_v2,
        sample_weight=paare.loc[trainieren, "produktgewicht"].to_numpy()
    )

    ergebnisse_v2.loc[pruefen, "p_anwesend"] = modell.predict_proba(
        features_v2.loc[pruefen, spalten]
    )[:, 1]

    modelle_v2[fold] = modell
    modell_spalten_v2[fold] = spalten

    r = ergebnisse_v2.loc[pruefen]
    brier_v1 = np.average(
        (r["p_anwesend_v1"] - r["vorhanden"]) ** 2,
        weights=r["produktgewicht"]
    )
    brier_v2 = np.average(
        (r["p_anwesend"] - r["vorhanden"]) ** 2,
        weights=r["produktgewicht"]
    )

    vergleich_v2.append({
        "Fold": fold,
        "Verglichene Paare": len(r),
        "Brier V1": brier_v1,
        "Brier V2": brier_v2,
        "Vorteil V2": brier_v1 - brier_v2
    })

ergebnisse_v2["auffaelligkeit"] = np.where(
    ergebnisse_v2["vorhanden"].eq(0),
    ergebnisse_v2["p_anwesend"],
    1 - ergebnisse_v2["p_anwesend"]
)

assert ergebnisse_v2["p_anwesend"].notna().equals(
    ergebnisse["p_anwesend"].notna()
)

display(pd.DataFrame(vergleich_v2).round(5))

In [0]:
alte_spitze = (
    ergebnisse.loc[ergebnisse["p_anwesend"].notna()]
    .sort_values("auffaelligkeit", ascending=False)
    .groupby("pruefrichtung", sort=False)
    .head(5)
)

fallvergleich_v2 = alte_spitze[[
    "erzeugnis", "fassung_id", "komponente",
    "vorhanden", "pruefrichtung"
]].copy()

fallvergleich_v2["Auffälligkeit V1"] = alte_spitze["auffaelligkeit"]
fallvergleich_v2["Auffälligkeit V2"] = (
    ergebnisse_v2.loc[alte_spitze.index, "auffaelligkeit"]
)
fallvergleich_v2["Abnahme"] = (
    fallvergleich_v2["Auffälligkeit V1"]
    - fallvergleich_v2["Auffälligkeit V2"]
)

display(fallvergleich_v2.round(4))

In [0]:
pflicht = [
    "erzeugnis", "fassung_id", "komponente",
    "vorhanden", "p_anwesend", "auffaelligkeit"
]

assert set(pflicht).issubset(ergebnisse_v2.columns), \
    f"Fehlende Spalten: {set(pflicht) - set(ergebnisse_v2.columns)}"


quelldaten = pd.DataFrame(sauber)

def haeufigster_text(texte):
    texte = texte.fillna("").astype(str).str.strip()
    return texte.value_counts().index[0]


produktnamen = (
    quelldaten.groupby("erzeugnis")["erzeugnis_txt"]
    .agg(haeufigster_text)
)

komponentennamen = (
    quelldaten.groupby("komponente")["komponente_txt"]
    .agg(haeufigster_text)
)


rangliste_v2 = ergebnisse_v2.loc[
    ergebnisse_v2["p_anwesend"].notna(), pflicht
].copy()

rangliste_v2["Produktname"] = (
    rangliste_v2["erzeugnis"].map(produktnamen)
)

rangliste_v2["Komponentenname"] = (
    rangliste_v2["komponente"].map(komponentennamen)
)

assert rangliste_v2["Produktname"].notna().all()
assert rangliste_v2["Komponentenname"].notna().all()


rangliste_v2["pruefrichtung"] = np.where(
    rangliste_v2["vorhanden"].eq(1),
    "möglicherweise zusätzlich",
    "möglicherweise fehlend"
)

schluessel = ["erzeugnis", "fassung_id", "komponente"]

bekannte_paare = pd.MultiIndex.from_frame(
    alte_spitze[schluessel]
)

rangliste_v2["bereits_in_V1_geprüft"] = (
    pd.MultiIndex.from_frame(rangliste_v2[schluessel])
    .isin(bekannte_paare)
)

rangliste_v2 = rangliste_v2.sort_values(
    "auffaelligkeit",
    ascending=False,
    kind="stable"
)

rangliste_v2["Rang_je_Richtung"] = (
    rangliste_v2.groupby("pruefrichtung").cumcount() + 1
)

spitzenliste_v2 = rangliste_v2.copy()

separat_v2 = ergebnisse_v2.loc[
    ergebnisse_v2["p_anwesend"].isna()
].copy()

display(
    spitzenliste_v2[[
        "pruefrichtung",
        "erzeugnis",
        "Produktname",
        "komponente",
        "Komponentenname",
        "auffaelligkeit",
        "vorhanden",
        "bereits_in_V1_geprüft",
        "fassung_id",
        "Rang_je_Richtung",
    ]].round(4)
)

print(
    "Separat zu prüfende Paare ohne CatBoost-Wert:",
    len(separat_v2)
)